# Benchmark of ODE Solvers Using 2D Diffusion Equation

This page shows the results of benchmarks of ODE solvers using the 2D diffusion equation whose spatial derivatives are discretized by RBF-FD method in the method of lines.
Dirichlet boundary condition and Neumann boundary condition are tested.


## Problem

### Equation

$$
\frac{\partial u}{\partial t} = \alpha \nabla^2 u
$$

- Diffusion coefficient: $\alpha = 0.1$

### Conditions

- Dirichlet boundary condition
  - The diffusion equation is solved in the range $[0, 1] \times [0, 1]$.
  - All boundaries have the Dirichlet boundary condition with $u = 0$.
  - The initial condition is given by $u(x, y, 0) = \sin(\pi x) \sin(\pi y)$.
- Neumann boundary condition
  - The diffusion equation is solved in the range $[0, 1] \times [0, 0.5]$.
  - Boundary at $y = 0.5$ has the Neumann boundary condition with $\frac{\partial u}{\partial y} = 0$.
  - Other boundaries have the Dirichlet boundary condition with $u = 0$.
  - The initial condition is given by $u(x, y, 0) = \sin(\pi x) \sin(\pi y)$.

### Spatial Discretization

- Number of nodes
  - Dirichlet boundary condition: 2500 nodes in the interior and 200 nodes on the boundary.
  - Neumann boundary condition: 2500 nodes in the interior and 200 nodes on the boundary.
- In the case of the Neumann boundary condition, boundary nodes are included directly in the discretization to create DAEs. In this case, only ODE solvers supporting DAEs are used.


## Results

In [ ]:
import num_collect_docs_utils.plot_common

num_collect_docs_utils.plot_common.set_common_configuration()

In [ ]:
# Load data
import gzip
import msgpack
import pandas

with gzip.open("data/diffusion2d_dirichlet.data", mode="rb") as file:
    results = msgpack.unpack(file)
data_frame = pandas.DataFrame(
    {
        "Boundary Condition": ["Dirichlet"] * len(results["solver_list"]),
        "Solver": results["solver_list"],
        "Err. Tol.": results["tolerance_list"],
        "Error Rate": results["error_rate_list"],
        "Time [sec]": results["time_list"],
    }
)

with gzip.open("data/diffusion2d_neumann.data", mode="rb") as file:
    results = msgpack.unpack(file)
data_frame = pandas.concat(
    [
        data_frame,
        pandas.DataFrame(
            {
                "Boundary Condition": ["Neumann"] * len(results["solver_list"]),
                "Solver": results["solver_list"],
                "Err. Tol.": results["tolerance_list"],
                "Error Rate": results["error_rate_list"],
                "Time [sec]": results["time_list"],
            }
        ),
    ],
    ignore_index=True,
)

In [ ]:
# Show plots
import plotly.express
import num_collect_docs_utils.plot_common

figure = plotly.express.line(
    data_frame,
    x="Time [sec]",
    y="Error Rate",
    color="Solver",
    line_dash="Solver",
    facet_row="Boundary Condition",
    hover_data=["Solver", "Err. Tol.", "Error Rate", "Time [sec]"],
    markers=True,
    log_x=True,
    log_y=True,
    title="Work-Error Diagram of ODE Solvers for 2D Diffusion Equation",
)
figure.update_layout(height=900)
num_collect_docs_utils.plot_common.show_figure(figure)

## Execution Environment

- CPU: Intel(R) Core(TM) Ultra 5 125H (14 cores, 18 threads)
- Memory: 32 GB
- Commit: `c2652a5f2b0d5eab9ee38c33e025c4729a3e2f5d`
- Commands:
  - `./build/Release/bin/bench_ode_diffusion2d_dirichlet results/`
  - `./build/Release/bin/bench_ode_diffusion2d_neumann results/`
